In [1]:
import os
import gymnasium as gym
import numpy as np
import pygame
import seaborn as sns
import torch
import torch as th
import torch.nn as nn
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import SubprocVecEnv
from torch.distributions import Categorical
from torch.nn import functional as F
# 切换工作目录到COMPASS项目主目录
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "../..")))

d:\Anaconda\envs\graphflow\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [3]:
import gymnasium as gym
import compass_env
from compass_env.envs.compass_parallel_env import CompassParallelEnv

env = gym.make("compass-highway-v2", render_mode="human")
env.reset()

找到 168 个.state.xml.gz文件


({'ego': array([ 0.0000000e+00,  0.0000000e+00,  6.6861674e-02,  0.0000000e+00,
          8.4641367e-01, -5.3252596e-01,  1.0000000e+00,  1.0000000e+00,
          1.0000000e+00,  1.0000000e+00,  3.0000000e+00, -2.5000000e-01,
          1.0755590e-01, -1.8137365e-04,  1.6000000e+00,  5.5000001e-01,
          1.0000000e+00,  2.2200000e-01,  0.0000000e+00,  0.0000000e+00,
          1.0000000e+00, -9.6152171e-02,  1.9735226e-02,  1.5477241e-02,
          9.9999998e-03, -4.3369442e-01, -9.9999998e-03,  1.0000000e-03,
          9.7200000e-01,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
          1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  0.0000000e+00,
          0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  1.0000000e+00,
          0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  1.0000000e+00,
          0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
          0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
          1.0000000e+00], dtype=float32),
  

In [2]:
import numpy as np
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
import compass_env
import gymnasium as gym

def make_env(rank: int, base_port: int = 12000, seed: int = 0):
    def _init():
        cfg = {
            "simulation": {
                "traci_port": base_port + rank,
                "traci_label": f"compass-r{rank}",
            },
            "case_num": -1,  # 或固定一个 case 便于复现
        }
        env = gym.make("compass-highway-v2", config=cfg, render_mode=None, verbose=False)
        env.reset(seed=seed + rank)
        return env
    return _init


n_envs = 4
env = DummyVecEnv([make_env(i) for i in range(n_envs)])

obs = env.reset()
print("reset ok")
# obs 是 vectorized 结构：Dict[str, np.ndarray]
print({k: v.shape for k, v in obs.items()})



找到 168 个.state.xml.gz文件
找到 168 个.state.xml.gz文件
找到 168 个.state.xml.gz文件
找到 168 个.state.xml.gz文件
reset ok
{'ego': (4, 53), 'veh': (4, 15, 21, 25)}


In [3]:
actions = np.random.uniform(-1, 1, size=(n_envs, 2)).astype(np.float32)
obs, reward, done, info = env.step(actions)

In [3]:
obs['ego'].shape, obs['veh'].shape

((4, 53), (4, 15, 21, 25))

In [6]:
obs['ego']

array([[ 0.00000000e+00,  0.00000000e+00,  1.63405612e-01,
         0.00000000e+00,  8.33257496e-01, -5.52885115e-01,
         1.00000000e+00,  1.00000000e+00,  1.00000000e+00,
         1.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.70660704e-01,  0.00000000e+00,  1.60000002e+00,
         5.00000000e-01,  1.00000000e+00,  2.22000003e-01,
         1.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         9.99999978e-03, -1.48655608e-01, -9.99999978e-03,
         2.14886349e-02,  9.99999978e-03,  9.99999978e-03,
         1.00000005e-03,  9.72000003e-01,  0.00000000e+00,
         1.00000000e+00,  1.00000000e+00,  1.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  1.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  1.00000000e+00,  0.00000000e+0

In [ ]:
class CustomCombinedExtractor(BaseFeaturesExtractor):
    """
    :param observation_space: (gym.Space)
    :param features_dim: (int) Number of features extracted.
        This corresponds to the number of unit for the last layer.
    """
    
    def __init__(self, observation_space: gym.spaces.Dict, **kwargs):
        # We do not know features-dim here before going over all the items,
        # so put something dummy for now. PyTorch requires calling
        # nn.Module.__init__ before adding modules
        super().__init__(observation_space, features_dim=1)

        extractors = {}

        total_concat_size = 0
        # We need to know size of the output of this extractor,
        # so go over all the spaces and compute output feature sizes
        for key, subspace in observation_space.spaces.items():
            print(key, subspace.shape, subspace)
            if key == "ego":
                # We will just downsample one channel of the image by 4x4 and flatten.
                # Assume the image is single-channel (subspace.shape[0] == 0)
                extractors[key] = nn.Sequential(nn.Flatten())
                total_concat_size += subspace.shape[0]
            elif key == "veh":
                # Run through a simple MLP
                extractors[key] = nn.Conv2d(subspace.shape[2], 128, kernel_size=(1,1), padding=0)
                total_concat_size += 128 * subspace.shape[0] * subspace.shape[1]

        self.extractors = nn.ModuleDict(extractors)

        # Update the features dim manually
        self._features_dim = total_concat_size

    def forward(self, observations) -> th.Tensor:
        encoded_tensor_list = []
        # self.extractors contain nn.Modules that do all the processing.
        for key, extractor in self.extractors.items():
            if key == "veh":
                x = observations[key]
                x = extractor(x.permute(0, 3, 1, 2)).permute(0, 2, 3, 1)
                encoded_tensor_list.append(x.flatten(start_dim=1))
            else:
                encoded_tensor_list.append(extractor(observations[key]))
        # Return a (B, self._features_dim) PyTorch tensor, where B is batch dimension.
        return th.cat(encoded_tensor_list, dim=1)
    
    

attention_network_kwargs = dict(
    in_size=5 * 15,
    embedding_layer_kwargs={"in_size": 7, "layer_sizes": [64, 64], "reshape": False},
    attention_layer_kwargs={"feature_size": 64, "heads": 2},
)
policy_kwargs = dict(
    features_extractor_class=CustomCombinedExtractor,
    features_extractor_kwargs=attention_network_kwargs,
)
model = PPO(
    "MultiInputPolicy",
    env,
    n_steps=512 // 4,
    batch_size=64,
    learning_rate=2e-3,
    policy_kwargs=policy_kwargs,
    verbose=2,
)

Using cuda device
ego (53,) Box(-inf, inf, (53,), float32)
veh (15, 21, 25) Box(-inf, inf, (15, 21, 25), float32)


In [11]:
model.predict(obs)

{'ego': tensor([[ 0.0000e+00,  0.0000e+00,  8.7531e-02,  0.0000e+00,  8.9810e-01,
         -4.3979e-01,  1.0000e+00,  1.0000e+00,  1.0000e+00,  1.0000e+00,
          0.0000e+00,  0.0000e+00,  2.1121e-01,  0.0000e+00,  1.6000e+00,
          5.0000e-01,  1.0000e+00,  2.2200e-01,  1.0000e+00,  0.0000e+00,
          0.0000e+00,  2.9645e-02,  1.0000e-02, -3.1313e-02,  1.0000e-02,
          1.0000e-02,  1.0000e-02,  1.0000e-03,  9.7200e-01,  0.0000e+00,
          1.0000e+00,  1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  9.4503e-02,  0.0000e+00,  9.8795e-01,
         -1.5480e-01,  1.0000e+00,  1.0000e+00,  1.0000e+00,  1.0000e+00,
          0.0000e+00,  0.0000e+00,  1.7066e-01,  0.0000

(array([[-0.19983475, -0.9141878 ],
        [ 0.578241  , -1.        ],
        [ 0.4592807 ,  1.        ],
        [-1.        ,  1.        ]], dtype=float32),
 None)

In [ ]:
env.close()
print("close ok")